In [1]:
#brightway25 packages
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import bw2analyzer as bwa
import polars as pl
from rapidfuzz import fuzz, process

from archetypes_re2020.utils import bw25_utils as bw25u
from archetypes_re2020.utils.utils import *

from archetypes_re2020.process.processes_db import *

01:01:18+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

In [3]:
list(bd.databases)

['ecoinvent-3.11-cutoff',
 'ecoinvent-3.11-biosphere',
 '2026-4-23-18h10_archetypes_RE2020_PRB',
 '2026-5-7-20h29_bernoulli_RE2020',
 '2026-5-7-20h29_archetypes_RE2020_MIB',
 '2026-8-9-22h39_processes_RE2020',
 '2026-8-9_consumptions_processes',
 '2026-8-9_composants_yc_consos',
 '2026-8-9-23h6_buildings_RE2020',
 '2026-8-10-0h50_buildings_cons_RE2020']

In [ ]:
imp = bi.ExcelImporter("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/interim/2026-01-13_bdd_interim_procedes_RE2020.xlsx")
data = getattr(imp, "data")

In [ ]:
def node_to_str_dict(node,with_unit = True) :
    attrs = [
        'nomenclature_inies_1', 'nomenclature_inies_2', 'nomenclature_inies_3',
        'nomenclature_inies_4', 'nomenclature_inies_5', 'unit'
        ]
    tmp = {attr : node[attr] for attr in attrs}
    if with_unit :
        tmp["unite_uf_txt"] = tmp["unit"]
    else :
        del tmp['unit']
        
    return str(tmp)

def ratio_to_node(node, col_nom) :
    s = node_to_str_dict(node)
    tmp = eval(col_nom)
    tmp.popitem()
    return fuzz.ratio(s,str(tmp))


In [ ]:
path = "/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/external/2026-2-4_process_to_model_grouped.xlsx"
df = pl.read_excel(path)

In [ ]:
for node in data :
    df = df.with_columns(
        pl.col("nomenclatures_UF_dve")
        .map_elements(
            lambda x: ratio_to_node(,x),
            return_dtype = pl.Float64
        ).alias("d0")
    )
    

In [ ]:
with pl.Config(tbl_width_chars=-1, fmt_str_lengths=400,tbl_cols=-1,tbl_rows=-1):
    print(data[0])
    print(df["nomenclatures_UF_dve","d0"].filter(pl.col("d0")== pl.col("d0").max()))

In [ ]:

def strip_with_none(s, char=""):
    if s is None:
        return "XX"
    else:
        return s.strip(char)

In [ ]:
import functools
import operator
from collections import defaultdict

def find_dve(data, excel_path):
    df = pl.read_excel(excel_path)
    # # Parse the JSON/dict column once and expand it
    df = df.with_columns(
        pl.col("nomenclatures_UF_dve")
        .map_elements(lambda x: eval(x), return_dtype=pl.Object)
        .alias("parsed_dict")
    )
    # Filter out rows where all values (except last) are None
    df = df.with_columns(
        pl.col("parsed_dict")
        .map_elements(
            lambda x: all([e is None for e in list(x.values())[:-1]]),
            return_dtype=pl.Boolean
        )
        .alias("to_drop")
    )
    df = df.filter(pl.col('to_drop') == False).drop("to_drop")
    # Extract dve value
    df = df.with_columns(
        pl.col("parsed_dict")
        .map_elements(lambda x: x.get("dve"), return_dtype=pl.Float64)
        .alias("dve")
    )

    df = df.drop("parsed_dict")
    # # Filter matching rows
    lens = []
    filter_attrs = [
        'nomenclature_inies_1', 'nomenclature_inies_2', 'nomenclature_inies_3',
        'nomenclature_inies_4', 'nomenclature_inies_5', 'unit'
    ]
    
    # Counter to track how many times each combination has been seen
    combination_counter = defaultdict(int)
    
    for node in data:
        df = df.with_columns(
            pl.col("nomenclatures_UF_dve")
            .map_elements(
                lambda x: ratio_to_node(node,x),
                return_dtype = pl.Float64
            ).alias("dist_to_node")
        )
        filters = (
            (pl.col("dist_to_node")>95) 
            & (pl.col("dist_to_node")==pl.col("dist_to_node").max())
            )

        tmp = df.filter(filters)
        if len(tmp) ==5:
            with pl.Config(tbl_width_chars=-1, fmt_str_lengths=400,tbl_cols=-1,tbl_rows=-1):
                print(tmp)
        lens.append(len(tmp))

        if len(tmp) > 0:
            # Create a hashable key from the combination of filter attributes
            combo_key = node_to_str_dict(node,with_unit=False)
            
            # Get the current counter for this combination
            a = combination_counter[combo_key]
            
            # Assign the dve value at index a (with bounds checking)
            if a < len(tmp):
                node["dve"] = tmp["dve"][a]
            else:
                # Handle case where we've exhausted all matches
                node["dve"] = tmp["dve"][-1]  # or None, or handle differently
            
            # Increment the counter for this combination
            combination_counter[combo_key] += 1
    
    print(f"DataFrame length: {len(df)}, Data length: {len(data)}, Match counts: {lens}")

In [ ]:
path = "/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/external/2026-2-4_process_to_model_grouped.xlsx"
find_dve(data, path)


In [ ]:
[d["dve"] for d in data]

In [ ]:
data[0]

In [ ]:
def update_node_codes(data):
    """
    Met à jour les codes de tous les nœuds au format 'nom_unité'.

    Args:
        database (bd.Database): Base de données brightway.
    """
    for node in data:
        node['code'] = "_".join([str(node['name']).lower(), str(node['unit'])])

In [ ]:
update_node_codes(imp)

In [ ]:
getattr(imp, "data")[0]

In [ ]:
for proc in data : 
    print(proc.get("code"))

In [ ]:
import polars as pl

df= pl.read_excel("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/external/2026-2-4_process_to_model_grouped.xlsx")

In [ ]:
with pl.Config(tbl_width_chars=-1, fmt_str_lengths=400,tbl_cols=-1):
    print(df[9])

In [ ]:
list(data[0])

In [ ]:
df["nomenclatures_UF_dve"][0]

In [ ]:
df = df.with_columns(
    pl.col("nomenclatures_UF_dve")
    .map_elements(
        lambda x : list(dict(list(eval(x).items())[:-1]).values()),
        return_dtype = pl.Object
    ).alias("nom_uf")
)

In [ ]:
import ast 

ast.literal.evals(data[0])

In [ ]:
l = [list({attr : node[attr].strip() for attr in ['nomenclature_inies_1',
 'nomenclature_inies_2',
 'nomenclature_inies_3',
 'nomenclature_inies_4',
 'nomenclature_inies_5','unit']}.values()) for node in data]

In [ ]:
list(df["nom_uf"][0].values()) == list(l[0].values())

In [ ]:
l

In [ ]:
res = []
for s in df["nom_uf"] :
    if s in l :
        pass
    else : 
        res.append(s)

In [ ]:
len(res)

In [ ]:
['Produits de construction',
  'Revêtements des sols et murs / peintures / produits de décoration',
  'Peintures, lasures et vernis, enduits de peintures (hors saturateurs)',
  'Peintures aqueuses',
  None,
  'm²'] in l

In [ ]:
['Produits de construction',
  'Revêtements des sols et murs / peintures / produits de décoration',
  'Peintures, lasures et vernis, enduits de peintures (hors saturateurs)',
  'Peintures aqueuses',
  None,
  'm²'] == ['Produits de construction',
  'Revêtements des sols et murs / peintures / produits de décoration ',
  'Peintures, lasures et vernis, enduits de peintures (hors saturateurs)',
  'Peintures aqueuses',
  None,
  'm²']

In [ ]:
['Produits de construction',
  'Revêtements des sols et murs / peintures / produits de décoration ',
  'Peintures, lasures et vernis, enduits de peintures (hors saturateurs)',
  'Peintures aqueuses',
  None,
  'm²'] in l

In [ ]:
res

In [2]:
bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

In [3]:
bd.databases

Databases dictionary with 9 object(s):
	2026-2-5-19h15_procedes_RE2020
	Condorcet_db
	ciogen_custom
	ciogen_fin_de_vie
	ciogen_ponts_production
	ciogen_production
	ciogen_transport
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff

In [4]:
fgdb = bd.Database("2026-2-5-19h15_procedes_RE2020")

In [7]:
a =fgdb.random()
a.as_dict()

{'categories': ('custom_process',),
 'code': 'plinthes / moulures_céramique_m_30',
 'comment': 'un commentaire',
 'location': 'FR',
 'nomenclature_inies_1': 'Produits de construction',
 'nomenclature_inies_2': 'Revêtements des sols et murs / peintures / produits de décoration',
 'nomenclature_inies_3': 'Plinthes / moulures',
 'nomenclature_inies_4': 'Céramique',
 'nomenclature_inies_5': None,
 'production amount': 1.0,
 'type': 'processwithreferenceproduct',
 'unit': 'meter',
 'name': 'Plinthes / moulures_Céramique',
 'worksheet name': 'Sheet1',
 'database': '2026-2-5-19h15_procedes_RE2020',
 'dve': 30.0,
 'id': 277864895900639233}

In [9]:
lca = bc.LCA(demand = {a:1.0},method=[m for m in bd.methods if "gwp" in str(m).lower()][0])

In [10]:
lca.lci()
lca.lcia()
lca.score

2.5051379128587703

In [4]:
prodb = bd.Database("2026-2-12-10h19_procedes_RE2020")

In [6]:
prodb.metadata

{'format': 'Excel spreadsheet',
 'date': '2025-10-01_11h39',
 'depends': ['ecoinvent-3.11-cutoff'],
 'backend': 'sqlite',
 'number': 383,
 'modified': '2026-02-12T10:21:47.514502',
 'geocollections': ['world'],
 'processed': '2026-02-12T16:15:10.742210',
 'searchable': True}

In [6]:
a = prodb.random()

In [12]:
inputs = [bd.get_node(database = e['input'][0],code = e['input'][1]) for e in a.technosphere()]

In [18]:
a, b = (1,2)

In [20]:
b

2

In [8]:
generate_database_name("PROCESSES")

'2026-3-11-19h21_processes_RE2020'

In [5]:
list(bd.databases)

['ecoinvent-3.11-biosphere',
 'ecoinvent-3.11-cutoff',
 'Condorcet_db',
 'ciogen_production',
 'ciogen_transport',
 'ciogen_fin_de_vie',
 'ciogen_custom',
 'ciogen_ponts_production',
 'bernou_test',
 '2026-2-12-10h19_procedes_RE2020',
 '2026-2-12-10h23_batiments_RE2020',
 '2026-2-12-10h51_bernoulli_RE2020',
 '2026-2-12-10h51_archetypes_RE2020_MI',
 '2026-2-12-15h17_archetypes_RE2020_BB',
 '2026-2-12-10h23_batiments_RE2020_wo_pump',
 '2026-2-26-15h42_buildings_RE2020',
 '2026-2-26-16h45_archetypes_RE2020_BB',
 '2026-3-12-13h52_processes_RE2020',
 '2026-3-13-17h6_processes_RE2020']

In [ ]:
del bd.databases['2026-3-11-20h2_processes_RE2020']

In [4]:
eidb = bd.Database("ecoinvent-3.11-cutoff")
prodb = bd.Database('2026-3-13-17h6_processes_RE2020')

In [15]:
a = prodb.search("dalles",limit = 100)[2]
a

'Dalles et prédalles_Béton plein armé' (m², FR, ('custom_process',))

In [16]:
a.as_dict()

{'categories': ('custom_process',),
 'code': 'dalles et prédalles_béton plein armé_m²_50',
 'comment': 'un commentaire',
 'location': 'FR',
 'nomenclature_inies_1': 'Produits de construction',
 'nomenclature_inies_2': 'Structure / maçonnerie / gros œuvre / charpente',
 'nomenclature_inies_3': 'Dalles et prédalles',
 'nomenclature_inies_4': 'Béton plein armé',
 'nomenclature_inies_5': None,
 'production amount': 1.0,
 'type': 'processwithreferenceproduct',
 'unit': 'm²',
 'name': 'Dalles et prédalles_Béton plein armé',
 'worksheet name': 'Sheet2',
 'database': '2026-3-13-17h6_processes_RE2020',
 'dve': 50.0,
 'id': 290878567518396417,
 'reference product': 'dalles et prédalles_béton plein armé_m²_50'}

In [17]:
list(a.edges())

[Exchange: -500 kilogram 'market for waste concrete' (kilogram, Europe without Switzerland, None) to 'Dalles et prédalles_Béton plein armé' (m², FR, ('custom_process',)),
 Exchange: 1.0 m² 'Dalles et prédalles_Béton plein armé' (m², FR, ('custom_process',)) to 'Dalles et prédalles_Béton plein armé' (m², FR, ('custom_process',)),
 Exchange: 0.2 cubic meter 'market for concrete, 30MPa' (cubic meter, RoW, None) to 'Dalles et prédalles_Béton plein armé' (m², FR, ('custom_process',)),
 Exchange: 20.0 kilogram 'market for reinforcing steel' (kilogram, GLO, None) to 'Dalles et prédalles_Béton plein armé' (m², FR, ('custom_process',))]

In [6]:
for a in prodb :
    try : 
        exch = [bd.get_node(database = e["input"][0], code=e["input"][1])["name"] for e in a.edges()]
        if "concrete slab" in exch :
            print("concrete slab in : ",a["name"])
    except Exception as e :
        print(e,a["name"])


 Fenêtres / portes fenêtres_Bois
 Menuiseries intérieures et extérieures / fermetures_Fenêtres de toit
 Fenêtres / portes fenêtres_PVC
 Fenêtres / portes fenêtres_Acier
 Fenêtres / portes fenêtres_Aluminium


In [5]:
node = prodb.search("Fenêtres / portes fenêtres_Bois")[0]

In [19]:
    glass_prod_input = eidb.search(GLASS_SEARCH)[0]
    waste_glass_input = eidb.search(WASTE_GLASS_SEARCH)[0]

    glass_prod_exch = {
        "input" : (glass_prod_input["database"],glass_prod_input["id"]),
        "type" : "processwithreferenceproduct",
        "amount" : 0, #quantité définie après
        "step" : "A1-A4" ,
        "unit" : "square meter"
    }
    glass_renew_exch = {
        "input" : (glass_prod_input["database"],glass_prod_input["id"]),
        "type" : "processwithreferenceproduct",
        "amount" : 0, #quantité définie après
        "step" : "B4" ,
        "unit" : "square meter"
    }

    waste_glass_exch = {
        "input" : (waste_glass_input["database"],waste_glass_input["id"]),
        "type" : "processwithreferenceproduct",
        "amount" : 0, #quantité définie après
        "step" : "C2-C4" ,
        "unit" : "square meter"
    }

In [16]:
glass_prod_exch

{'input': ('ecoinvent-3.11-cutoff', 269493679624118274),
 'type': 'processwithreferenceproduct',
 'amount': 0,
 'step': 'A1-A4',
 'unit': 'square meter'}

In [17]:
prod_exch = [
    e for e in node.technosphere() if e["step"]=="A1-A4"
    ][0]

prod_exch

Exchange: 0.19999999999999996 square meter 'market for window frame, wood, U=1.5 W/m2K' (square meter, GLO, None) to 'Fenêtres / portes fenêtres_Bois' (m², FR, ('custom_process',))

In [20]:
if prod_exch["amount"] < 1 :
    glass_prod_exch["amount"] = 1-prod_exch["amount"]
elif prod_exch["amount"] == 1 :
    glass_prod_exch["amount"] = DEFAULT_GLASS_RATE

In [21]:
glass_prod_exch

{'input': ('ecoinvent-3.11-cutoff', 269493679624118274),
 'type': 'processwithreferenceproduct',
 'amount': 0.8,
 'step': 'A1-A4',
 'unit': 'square meter'}

In [24]:
list(node.technosphere())

[Exchange: 0.19999999999999996 square meter 'market for window frame, wood, U=1.5 W/m2K' (square meter, GLO, None) to 'Fenêtres / portes fenêtres_Bois' (m², FR, ('custom_process',)),
 Exchange: 0.19999999999999996 square meter 'market for window frame, wood, U=1.5 W/m2K' (square meter, GLO, None) to 'Fenêtres / portes fenêtres_Bois' (m², FR, ('custom_process',)),
 Exchange: -0.3999999999999999 square meter 'market for used window frame, wood' (square meter, GLO, None) to 'Fenêtres / portes fenêtres_Bois' (m², FR, ('custom_process',))]

In [23]:
prod_edge = node.new_edge(**glass_prod_exch)
prod_edge.save()

In [18]:
hp_prod = eidb.search(HEAT_PUMP_SEARCH)

In [38]:
eidb.search("market for used refrigeration machine, R134a as refrigerant, GLO")[0]

'market for used refrigeration machine, R134a as refrigerant' (unit, GLO, None)

In [36]:
l = [a for a in eidb if "brine-water" in str(a).lower()]
l

['heat pump production, brine-water, 10kW' (unit, RoW, None),
 'heat production, borehole heat exchanger, brine-water heat pump 10kW' (megajoule, CH, None),
 'market for heat pump, brine-water, 10kW' (unit, GLO, None),
 'heat production, borehole heat exchanger, brine-water heat pump 10kW' (megajoule, RoW, None),
 'heat production, borehole heat exchanger, brine-water heat pump 10kW' (megajoule, Europe without Switzerland, None),
 'heat pump production, brine-water, 10kW' (unit, CH, None)]

In [25]:
eidb

Brightway2 SQLiteBackend: ecoinvent-3.11-cutoff

In [5]:
heat_pump_unit_process(prodb,eidb)

Changement des procédés PAC:   0%|          | 0/367 [00:00<?, ?it/s]

Changement des procédés PAC: 100%|██████████| 367/367 [00:00<00:00, 376.50it/s]


In [9]:
a = [a for a in prodb if "pompes à chaleur" in str(a).lower()][4]
list(a.edges())

[Exchange: 1.0 unité 'Pompes à chaleur collectives/tertiaires_Air/Air' (unité, FR, ('custom_process',)) to 'Pompes à chaleur collectives/tertiaires_Air/Air' (unité, FR, ('custom_process',)),
 Exchange: 0.1 unit 'market for heat pump, brine-water, 10kW' (unit, GLO, None) to 'Pompes à chaleur collectives/tertiaires_Air/Air' (unité, FR, ('custom_process',)),
 Exchange: 0.8 unit 'market for heat pump, brine-water, 10kW' (unit, GLO, None) to 'Pompes à chaleur collectives/tertiaires_Air/Air' (unité, FR, ('custom_process',)),
 Exchange: 0.9 unit 'market for used refrigeration machine, R134a as refrigerant' (unit, GLO, None) to 'Pompes à chaleur collectives/tertiaires_Air/Air' (unité, FR, ('custom_process',))]

In [10]:
a["dve"]

22.0

In [11]:
a.as_dict()

{'categories': ('custom_process',),
 'code': 'pompes à chaleur collectives/tertiaires_air/air_unité_22',
 'comment': 'un commentaire',
 'location': 'FR',
 'nomenclature_inies_1': 'Équipements électriques, électroniques et de génie climatique',
 'nomenclature_inies_2': 'Equipements de génie climatique',
 'nomenclature_inies_3': 'Chauffage et/ou rafraîchissement et/ou production d’eau chaude sanitaire',
 'nomenclature_inies_4': 'Pompes à chaleur collectives/tertiaires',
 'nomenclature_inies_5': 'Air/Air',
 'production amount': 1.0,
 'type': 'processwithreferenceproduct',
 'unit': 'unité',
 'name': 'Pompes à chaleur collectives/tertiaires_Air/Air',
 'worksheet name': 'Sheet2',
 'database': '2026-3-13-17h6_processes_RE2020',
 'dve': 22.0,
 'id': 290878567614865409,
 'reference product': 'pompes à chaleur collectives/tertiaires_air/air_unité_22'}

In [7]:
fgdb = bd.Database('2026-8-9_composants_yc_consos')


In [8]:
len(fgdb)

375

In [15]:
a = fgdb.search("eau")

In [16]:
for b in a :
    print(list(b.edges()))

[Exchange: 1 unit 'market for heat pump, 30kW' (unit, GLO, None) to 'Chauffe-eau individuel à accumulation_Chauffe-eau thermodynamique' (unité, FR, ('custom_process',)), Exchange: 2 unit 'market for heat pump, 30kW' (unit, GLO, None) to 'Chauffe-eau individuel à accumulation_Chauffe-eau thermodynamique' (unité, FR, ('custom_process',)), Exchange: -3 unit 'market for used domestic refrigerator' (unit, GLO, None) to 'Chauffe-eau individuel à accumulation_Chauffe-eau thermodynamique' (unité, FR, ('custom_process',)), Exchange: 1.0 unité 'Chauffe-eau individuel à accumulation_Chauffe-eau thermodynamique' (unité, FR, ('custom_process',)) to 'Chauffe-eau individuel à accumulation_Chauffe-eau thermodynamique' (unité, FR, ('custom_process',))]
[Exchange: 1 cubic meter 'Consommations_eau' (cubic meter, FR, ('custom_process',)) to 'Consommations_eau' (cubic meter, FR, ('custom_process',)), Exchange: 1000 kilogram 'market for tap water' (kilogram, Europe without Switzerland, None) to 'Consommatio